In [ ]:
from pathlib import Path
import numpy as np
import open3d as o3d

from plant_shape_analysis.dataloaders.trackplant3D import LeafSequencesDataset
from plant_shape_analysis.data_preprocessing.leaf_meshing import mesh_poisson_o3d, mesh_ball_pivoting_o3d, mesh_leaf

from plant_shape_analysis.reconstruction import classical_methods

%load_ext autoreload
%autoreload 2

## Mesh example single leaf

In [ ]:
dataset_path = Path("../data/TrackPlant3D")
leaf_dataset = LeafSequencesDataset(
    dataset_path, 
    min_timepoints=1, # we get all leaves
)

In [ ]:
maize_leaves = leaf_dataset.get_timeseries_by_crop('maize')
len(maize_leaves)

In [ ]:
seq_id = 4
timepoint = 0
leaf = maize_leaves[seq_id]['timepoints'][timepoint]
leaf

In [ ]:
file_name = leaf['file_path'].name.replace(".txt", f"_leaf{leaf['leaf_id']}_dense.ply")
file_name

In [ ]:
# pcd = o3d.geometry.PointCloud()
# pcd.points = o3d.utility.Vector3dVector(leaf['dense_points'])
# o3d.io.write_point_cloud(f"../data/TrackPlant3D/examples/single_leafs/maize/{file_name}", pcd)

In [ ]:
pcd, mesh = mesh_leaf(leaf, method="ball_pivoting", backend="pymeshlab")

In [ ]:
# o3d.visualization.draw_geometries(
# [mesh],
# point_show_normal=True,
# window_name="Leaf Point Cloud and normals",
# )

## Extract all dense leaves

In [ ]:
dataset_path = Path("../data/TrackPlant3D")
leaf_dataset = LeafSequencesDataset(
    dataset_path, 
    min_timepoints=1, # we get all leaves
)

In [ ]:
# for seq in leaf_dataset:
#     for scan in seq['timepoints']:
#         if scan['dense_points'] is not None:
#             # save leaf
#             crop = ''.join([c for c in seq['sequence_name'].split('_')[0] if c.isalpha()])
#             file_name = scan['file_path'].name.replace(".txt", f"_leaf{scan['leaf_id']}_dense.ply")
#             pcd = o3d.geometry.PointCloud()
#             pcd.points = o3d.utility.Vector3dVector(scan['dense_points'])
#             save_dir = Path(f"../data/TrackPlant3D/dense_leaves/{crop}")
#             save_dir.mkdir(parents=True, exist_ok=True)
#             o3d.io.write_point_cloud(save_dir / file_name, pcd)

## Classical methods for surface reconstruction

In [3]:
# Load and visualize a sample leaf point cloud
sample_leaf_path = Path("../data/TrackPlant3D/dense_leaves/maize/1_maize_control_plant1_D00_leaf2_dense.ply")
leaf_pcd = o3d.io.read_point_cloud(str(sample_leaf_path))
leaf_pcd.estimate_normals()
leaf_pcd.orient_normals_to_align_with_direction()
leaf_pcd.orient_normals_consistent_tangent_plane(k=50)
leaf_np = np.asarray(leaf_pcd.points)
point_cloud = np.hstack([leaf_np, np.asarray(leaf_pcd.normals)])
print(f"Loaded leaf point cloud with {leaf_np.shape[0]} points.")
o3d.visualization.draw_geometries([leaf_pcd])

Loaded leaf point cloud with 34789 points.


### Apha shapes

In [ ]:
mesh_alpha = classical_methods.alpha_shape_reconstruction(leaf_np)
o3d.visualization.draw_geometries([mesh_alpha]) # -> Not bad but has holes

### Bezier

In [ ]:
mesh_bezier = classical_methods.bezier_surface_reconstruction(leaf_np)
o3d.visualization.draw_geometries([mesh_bezier]) # -> Has some issues, needs tuning

### B-spline

In [ ]:
mesh_bspline = classical_methods.bspline_surface_reconstruction(leaf_np)
o3d.visualization.draw_geometries([mesh_bspline]) # -> Looks better than bezier, but has some singularities

### Delaunay

In [ ]:
mesh_delaunay = classical_methods.triangulate_delaunay(leaf_np)
o3d.visualization.draw_geometries([mesh_delaunay])

### Poisson

In [ ]:
mesh_poisson = classical_methods.poisson_reconstruction(leaf_np, depth=9)
o3d.visualization.draw_geometries([mesh_poisson])
mesh_poisson.get_surface_area()

### Ball pivoting

In [6]:
mesh_bpa = classical_methods.ball_pivoting_reconstruction(leaf_np, radius_multiplier=5.0)
o3d.visualization.draw_geometries([mesh_bpa])
mesh_bpa.get_surface_area()

KeyboardInterrupt: 

## Compute traits

### Area

Alpha shapes:

In [ ]:
# Option 1: open3d built-in area
mesh_alpha.get_surface_area()

In [ ]:
# Option 2: manual area computation
vertices_alpha = np.asarray(mesh_alpha.vertices)
triangles_alpha = np.asarray(mesh_alpha.triangles)

area_alpha_manual = classical_methods.compute_mesh_area(vertices_alpha, triangles_alpha)
float(area_alpha_manual)

B-splines:

In [ ]:
mesh_bspline.get_surface_area()

In [ ]:
vertices_bspline = np.asarray(mesh_bspline.vertices)
triangles_bspline = np.asarray(mesh_bspline.triangles)

area_bspline_manual = classical_methods.compute_mesh_area(vertices_bspline, triangles_bspline)
float(area_bspline_manual)

Bezier:

In [ ]:
mesh_bezier.get_surface_area()

In [ ]:
vertices_bezier = np.asarray(mesh_bezier.vertices)
triangles_bezier = np.asarray(mesh_bezier.triangles)

area_bezier_manual = classical_methods.compute_mesh_area(vertices_bezier, triangles_bezier)
float(area_bezier_manual)

## TODOs
- [x] Save all leaves
- [x] Try different basic methods to fit the leaves
- [ ] Inspect manually which leaves are good for GT, discard noisy ones